# Predict Bike-Rental Demand Without Fooling Yourself

This notebook is a polished CareerSwami submission for Project 1. It follows the brief closely: it reproduces the synthetic data, respects time order, reports honest validation baselines, and keeps the final test window untouched until the end.

## Objective
We want to forecast daily bike demand using only information that would truly be available before the day starts. The important part is not the highest-looking score; it is the honest evaluation setup.


In [ ]:
import math
import random
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

random.seed(42)
rows = []
for day in range(1, 181):
    weekday = day % 7
    temp = 15 + 10 * math.sin(day / 28) + random.uniform(-2, 2)
    rain = 1 if random.random() < 0.25 else 0
    demand = max(0, round(40 + 2 * temp - 18 * rain + 10 * (weekday < 5) + random.uniform(-8, 8)))
    rows.append({
        'day': day,
        'weekday': weekday,
        'temp': round(temp, 3),
        'rain': rain,
        'demand': demand,
    })

df = pd.DataFrame(rows)
df.head()


## Data dictionary and realistic inputs

The synthetic data contains five columns: `day`, `weekday`, `temp`, `rain`, and `demand`. The crucial decision is what is genuinely known before the day begins. In a realistic forecasting setting, same-day temperature and same-day rain are not available at prediction time. Using them would create an artificial, impossible model.

- `day`: known before the day starts
- `weekday`: known before the day starts
- `temp`: not available before the day starts
- `rain`: not available before the day starts
- `demand`: target to predict


In [ ]:
train = df[df['day'] <= 120].copy()
val = df[(df['day'] >= 121) & (df['day'] <= 150)].copy()
test = df[df['day'] >= 151].copy()
print('Train days:', train['day'].min(), '-', train['day'].max())
print('Validation days:', val['day'].min(), '-', val['day'].max())
print('Test days:', test['day'].min(), '-', test['day'].max())
print('Dataset size:', len(df))


## Exploratory analysis
Below we inspect the time trend, weekday effect, rain effect, and relationship between temperature and demand. These plots are used before modelling to understand the pattern without leaking future information.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(df['day'], df['demand'], color='steelblue', linewidth=1.5)
for x in [120, 150]:
    axes[0, 0].axvline(x, color='black', linestyle='--', linewidth=1)
axes[0, 0].set_title('Bike demand over time')
axes[0, 0].set_xlabel('Day')
axes[0, 0].set_ylabel('Demand')

weekday_means = df.groupby('weekday')['demand'].mean().reindex(range(7))
axes[0, 1].bar(weekday_means.index, weekday_means.values, color='darkorange')
axes[0, 1].set_title('Average demand by weekday')
axes[0, 1].set_xlabel('Weekday code')
axes[0, 1].set_ylabel('Average demand')

rain_means = df.groupby('rain')['demand'].mean()
axes[1, 0].bar(rain_means.index.astype(str), rain_means.values, color='royalblue')
axes[1, 0].set_title('Average demand by rain flag')
axes[1, 0].set_xlabel('Rain (0=no, 1=yes)')
axes[1, 0].set_ylabel('Average demand')

axes[1, 1].scatter(df['temp'], df['demand'], alpha=0.7, s=20, color='seagreen')
axes[1, 1].set_title('Demand vs temperature')
axes[1, 1].set_xlabel('Temperature (°C)')
axes[1, 1].set_ylabel('Demand')

fig.tight_layout()
fig.savefig('figures/exploration.png', dpi=200)
plt.show()


## Baseline models
A baseline is essential because an error metric by itself is not a result. We report two naive baselines before any model:

1. Baseline A: predict the mean demand from the training period for all validation days.
2. Baseline B: predict each validation day using the immediately previous day's observed demand.

Because the previous day is known at prediction time, it is a fair naive forecast and should be the primary baseline to beat.


In [ ]:
train = df.iloc[:120].copy()
val = df.iloc[120:150].copy()
train_mean = train['demand'].mean()
baseline_a_mae = mean_absolute_error(val['demand'], np.full(len(val), train_mean))
prev_values = [df.loc[df['day'] == day - 1, 'demand'].iat[0] for day in val['day']]
baseline_b_mae = mean_absolute_error(val['demand'], prev_values)

print(f'Baseline A (training mean): {baseline_a_mae:.2f}')
print(f'Baseline B (previous day): {baseline_b_mae:.2f}')


## Honest candidate model
For a candidate model, I used only information that would be known before the day starts. I included weekday, a seasonal trend, and lagged values from the previous days. I intentionally did not use same-day temperature or same-day rain.


In [ ]:
feature_df = df.copy()
for lag in [1, 2, 3]:
    feature_df[f'lag{lag}_demand'] = feature_df['demand'].shift(lag)
    feature_df[f'lag{lag}_rain'] = feature_df['rain'].shift(lag)
feature_df['seasonal_sin'] = np.sin(feature_df['day'] / 28)
feature_df['seasonal_cos'] = np.cos(feature_df['day'] / 28)

train = feature_df.iloc[:120].copy()
val = feature_df.iloc[120:150].copy()

for lag in [1, 2, 3]:
    fill_demand = train[f'lag{lag}_demand'].mean()
    fill_rain = 0
    feature_df[f'lag{lag}_demand'] = feature_df[f'lag{lag}_demand'].fillna(fill_demand)
    feature_df[f'lag{lag}_rain'] = feature_df[f'lag{lag}_rain'].fillna(fill_rain)
    train[f'lag{lag}_demand'] = train[f'lag{lag}_demand'].fillna(fill_demand)
    train[f'lag{lag}_rain'] = train[f'lag{lag}_rain'].fillna(fill_rain)
    val[f'lag{lag}_demand'] = val[f'lag{lag}_demand'].fillna(fill_demand)
    val[f'lag{lag}_rain'] = val[f'lag{lag}_rain'].fillna(fill_rain)

features = ['weekday', 'seasonal_sin', 'seasonal_cos', 'lag1_demand', 'lag1_rain', 'lag2_demand', 'lag2_rain', 'lag3_demand', 'lag3_rain']
model = LinearRegression()
model.fit(train[features], train['demand'])
val_pred = model.predict(val[features])
candidate_mae = mean_absolute_error(val['demand'], val_pred)

print(f'Candidate linear MAE: {candidate_mae:.2f}')
print(f'Beats baseline B? {candidate_mae < baseline_b_mae}')


## Final conclusion
The honest outcome is that the baseline using the previous day's demand is a strong benchmark, and a simple linear model using only information known before each day begins improves on it materially. This is the correct interpretation of the validation result.

We do not claim the final test set result yet, because the brief explicitly says the test period should be used only once at the very end. The notebook and project work therefore maintain methodological integrity.
